# Pseudo-Real-Time Data Synthesizer

In macroeconomic forecasting, compiling historical *"Vintages"* (what a data matrix looked like on a specific day in history) can be incredibly tedious. Often, we just have a modern CSV with fully-revised data up to today.

This tool allows a user to supply a single modern dataset alongside a **Rules Dictionary**. The program systematically scrubs backwards, replacing historical data with `NaN`s mathematically exactly as it would have appeared at the cut-off date.

In [1]:
import pandas as pd
import numpy as np
from dfm_sp import VintageMaker, WeekdayRule, FixedDayRule

# 1. Load your standard perfectly complete, modern dataset
dates = pd.date_range(start='2023-01-01', end='2023-12-01', freq='MS')

full_dataset = pd.DataFrame({
    'PAYEMS': np.random.normal(200, 50, len(dates)),      # Nonfarm Payrolls
    'INDPRO': np.random.normal(100, 2, len(dates)),       # Industrial Production
    'RETAIL': np.random.normal(500, 10, len(dates))       # Retail Sales
}, index=dates)

print("--- Modern, Omniscient Dataset ---")
display(full_dataset.tail())

--- Modern, Omniscient Dataset ---


,PAYEMS,INDPRO,RETAIL
2023-08-01,292.279244,101.089571,495.478964
2023-09-01,108.005627,99.335472,498.237988
2023-10-01,158.891560,104.029432,494.118601
2023-11-01,228.217027,97.944735,490.381786
2023-12-01,168.636476,100.225582,487.723482


### Defining Publication Rules
Because each agency releases data differently, we map rules for when the mathematical limit crossed.

In [2]:
rules = {
    # Nonfarm Payrolls: Published 1st Friday of the *next* month.
    # (e.g., September data is released on the 1st Friday of October)
    'PAYEMS': WeekdayRule(weekday=4, n=1, reference_lag_months=1), 
    
    # Industrial Production: Published ~15th of the *next* month.
    'INDPRO': FixedDayRule(publish_day=15, reference_lag_months=1),
    
    # Retail Sales: Published ~15th of the *next* month as well.
    'RETAIL': FixedDayRule(publish_day=15, reference_lag_months=1)
}

generator = VintageMaker(rules_dict=rules)

### Running the Synthesizer
Let's test what an economist running our model on **November 10th, 2023** would see in their matrix.

- **PAYEMS (Oct Data)**: Released on Nov 3rd (1st Friday). Valid! Kept.
- **INDPRO (Oct Data)**: Releases on Nov 15th. Invalid! Must be scrubbed to NaN.
Notice the ragged-edge "tail" automatically mathematical forms.

In [3]:
# The target date we are simulating
simulated_date = "2023-11-10"

vintage_df = generator(full_dataset, target_vintage_date=simulated_date)

print(f"--- Filtered Vintage Matrix exactly on: {simulated_date} ---")
display(vintage_df.tail())

--- Filtered Vintage Matrix exactly on: 2023-11-10 ---


,PAYEMS,INDPRO,RETAIL
2023-08-01,292.279244,101.089571,495.478964
2023-09-01,108.005627,99.335472,498.237988
2023-10-01,158.891560,NaN,NaN
2023-11-01,NaN,NaN,NaN
2023-12-01,NaN,NaN,NaN
